# MIP Guatemala 2013 — exploración reproducible

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JA-Osorio/mip-guatemala-2013-reproducible/blob/main/04_reproduccion_python/cuaderno_exploracion_mip_2013.ipynb)

Este cuaderno explica cómo pasar de los CSV validados a un análisis insumo-producto. La fuente de verdad computacional es `reproducir_mip_guatemala_2013.py`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd

REPO_URL = 'https://github.com/JA-Osorio/mip-guatemala-2013-reproducible.git'
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path('/content/mip-guatemala-2013-reproducible')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd()
    if not (ROOT / '02_resultados_y_diccionario').exists():
        candidate = ROOT.parent
        if (candidate / '02_resultados_y_diccionario').exists():
            ROOT = candidate
SRC = ROOT / '04_reproduccion_python' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
RESULTS = ROOT / '02_resultados_y_diccionario'
ROOT

## Metadatos y controles

In [ ]:
metadata = json.loads((RESULTS / 'metadatos_dataset.json').read_text(encoding='utf-8'))
controls = pd.read_csv(ROOT / '05_verificacion' / 'controles_reproduccion.csv')
display(pd.Series(metadata, name='valor').to_frame())
display(controls[['control_id', 'description', 'status', 'value', 'tolerance']])

## Carga de matrices

In [ ]:
def load_matrix(name):
    frame = pd.read_csv(RESULTS / 'matrices' / name)
    return frame[['codigo', 'producto']], frame.iloc[:, 2:].to_numpy(float)

products, Z_d = load_matrix('Z_domestica_2013.csv')
_, Z_m = load_matrix('Z_importada_2013.csv')
_, A_d = load_matrix('A_domestica_2013.csv')
_, A_m = load_matrix('A_importada_2013.csv')
_, L_d = load_matrix('Leontief_domestica_2013.csv')
{'Z_domestica': Z_d.shape, 'Z_importada': Z_m.shape, 'A_domestica': A_d.shape, 'A_importada': A_m.shape, 'Leontief': L_d.shape}

## Verificación matricial independiente

In [ ]:
residuo = (np.eye(152) - A_d) @ L_d - np.eye(152)
pd.Series({
    'suma_Z_domestica': Z_d.sum(),
    'suma_Z_importada': Z_m.sum(),
    'residuo_maximo_inversa': np.abs(residuo).max(),
})

## Indicadores para análisis IO

El repositorio entrega multiplicadores de producción, encadenamientos normalizados y requerimientos de importación, valor agregado y empleo. Las importaciones se tratan como fugas del circuito doméstico.

In [ ]:
from mip_gt.analysis import io_indicator_frame, safe_coefficient

primary = pd.read_csv(RESULTS / 'vectores' / 'coeficientes_primarios_2013.csv')
output = pd.read_csv(RESULTS / 'vectores' / 'produccion_y_utilizacion_2013.csv')
indicators = io_indicator_frame(
    codes=products['codigo'],
    labels=products['producto'],
    a_imported=A_m,
    leontief_domestic=L_d,
    value_added_coefficients=primary['valor_agregado_bruto'],
    employment_coefficients=safe_coefficient(
        output['puestos_trabajo'].to_numpy(float),
        output['produccion_precios_basicos'].to_numpy(float),
    ),
)
published = pd.read_csv(RESULTS / 'indicadores_io_2013.csv')
numeric = indicators.columns[2:]
assert np.allclose(indicators[numeric], published[numeric], rtol=1e-12, atol=1e-12)
indicators.nlargest(10, 'multiplicador_produccion_domestica')

## Choque de demanda final parametrizable

Cambie `PRODUCTO_CHOQUE` y `MONTO_MILLONES_Q`. El ejemplo calcula producción doméstica, importaciones, valor agregado y empleo asociados a un aumento exógeno de demanda final.

In [ ]:
PRODUCTO_CHOQUE = 'P001'
MONTO_MILLONES_Q = 1.0

shock = np.zeros(len(products))
j = products.index[products['codigo'].eq(PRODUCTO_CHOQUE)][0]
shock[j] = MONTO_MILLONES_Q
delta_x = L_d @ shock
v = primary['valor_agregado_bruto'].to_numpy(float)
e = safe_coefficient(
    output['puestos_trabajo'].to_numpy(float),
    output['produccion_precios_basicos'].to_numpy(float),
)
pd.Series({
    'producto_choque': products.loc[j, 'producto'],
    'demanda_final_millones_Q': shock.sum(),
    'produccion_domestica_millones_Q': delta_x.sum(),
    'importaciones_millones_Q': (A_m @ delta_x).sum(),
    'valor_agregado_millones_Q': v @ delta_x,
    'puestos_trabajo': e @ delta_x,
})